<a href="https://colab.research.google.com/github/renaljono249-stack/uts_2501010148/blob/main/uts_dam_revisi_dataset_kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Q1.1 — Upload dan ekstraksi dataset Kaggle
# Dataset: https://www.kaggle.com/datasets/saufan/dataset-pariwisata-yogyakarta
import os
import zipfile
from google.colab import files

BASE_DIR = "/content/proyek_uts_pariwisata"
FOLDER_RAW = os.path.join(BASE_DIR, "data", "raw")
FOLDER_PROCESSED = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(FOLDER_RAW, exist_ok=True)
os.makedirs(FOLDER_PROCESSED, exist_ok=True)

print("Silakan upload file ZIP Dataset Pariwisata Yogyakarta dari Kaggle.")
uploaded = files.upload()
if not uploaded:
    raise ValueError("Belum ada file ZIP yang diunggah.")

nama_zip = next(iter(uploaded))
if not zipfile.is_zipfile(nama_zip):
    raise ValueError("File yang diunggah bukan ZIP. Unduh dataset Kaggle dalam format ZIP.")

with zipfile.ZipFile(nama_zip, "r") as zip_ref:
    anggota_csv = [
        n for n in zip_ref.namelist()
        if n.lower().endswith(".csv") and not n.startswith("__MACOSX/")
    ]
    zip_ref.extractall(FOLDER_RAW)

print("ZIP diekstrak ke:", FOLDER_RAW)
print("CSV yang ditemukan:")
for nama in anggota_csv:
    print("-", nama)

csv_wajib = {"tour.csv", "tour_rating.csv", "user.csv"}
csv_tersedia = {os.path.basename(n).lower() for n in anggota_csv}
if not csv_wajib.issubset(csv_tersedia):
    raise FileNotFoundError(
        "Isi ZIP belum lengkap. Dibutuhkan: tour.csv, tour_rating.csv, dan user.csv. "
        f"Ditemukan: {sorted(csv_tersedia)}"
    )


Silakan upload file ZIP Dataset Pariwisata Yogyakarta dari Kaggle.


Saving archive (5).zip to archive (5).zip
ZIP diekstrak ke: /content/proyek_uts_pariwisata/data/raw
CSV yang ditemukan:
- DATASET PARIWISATA YOGYAKARTA/tour.csv
- DATASET PARIWISATA YOGYAKARTA/tour_rating.csv
- DATASET PARIWISATA YOGYAKARTA/user.csv


In [2]:
# Google Drive tidak wajib. Sel ini sengaja tidak melakukan mount agar Runtime → Run all berjalan tanpa prompt tambahan.
print("Folder kerja proyek:", BASE_DIR)


Folder kerja proyek: /content/proyek_uts_pariwisata


In [3]:
# Q1.2 — Membaca tiga dataset Kaggle
import pandas as pd
import os
import glob

folder_raw = FOLDER_RAW

def cari_file_csv(nama_file):
    hasil = glob.glob(os.path.join(folder_raw, "**", nama_file), recursive=True)
    if not hasil:
        raise FileNotFoundError(f"{nama_file} tidak ditemukan di {folder_raw}.")
    return hasil[0]

file_tour = cari_file_csv("tour.csv")
file_rating = cari_file_csv("tour_rating.csv")
file_user = cari_file_csv("user.csv")

# Dataset Kaggle menggunakan CSV; utf-8-sig membantu membaca BOM bila ada.
df_tour = pd.read_csv(file_tour, encoding="utf-8-sig")
df_rating = pd.read_csv(file_rating, encoding="utf-8-sig")
df_user = pd.read_csv(file_user, encoding="utf-8-sig")

print("DATASET BERHASIL DIBACA")
for nama, data in [("tour", df_tour), ("tour_rating", df_rating), ("user", df_user)]:
    print(f"\\n{nama}: {data.shape[0]:,} baris × {data.shape[1]} kolom")
    display(data.head())

print("\\nPemeriksaan relasi kunci:")
print("Place_Id unik di tour:", df_tour["Place_Id"].nunique())
print("Rating dengan Place_Id yang tidak ditemukan:",
      (~df_rating["Place_Id"].isin(df_tour["Place_Id"])).sum())
print("Rating dengan User_Id yang tidak ditemukan:",
      (~df_rating["User_Id"].isin(df_user["User_Id"])).sum())


DATASET BERHASIL DIBACA
\ntour: 126 baris × 11 kolom


,Place_Id,Place_Name,Description,Category,City,Price,Rating,Time_Minutes,Coordinate,Latitude,Longitude
0,85.0,Taman Pintar Yogyakarta,Taman Pintar Yogyakarta (bahasa Jawa: Hanacara...,Taman Hiburan,Yogyakarta,6000,4.5,120.0,"{'lat': -7.800671500000001, 'lng': 110.3676551}",-7.800671,110.367655
1,86.0,Keraton Yogyakarta,Keraton Ngayogyakarta Hadiningrat atau Keraton...,Budaya,Yogyakarta,15000,4.6,NaN,"{'lat': -7.8052845, 'lng': 110.3642031}",-7.805284,110.364203
2,87.0,Sindu Kusuma Edupark (SKE),Sindu Kusuma Edupark (SKE) merupakan sebuah de...,Taman Hiburan,Yogyakarta,20000,4.2,120.0,"{'lat': -7.767297300000001, 'lng': 110.3542486}",-7.767297,110.354249
3,88.0,Museum Benteng Vredeburg Yogyakarta,Museum Benteng Vredeburg (bahasa Jawa: ???????...,Budaya,Yogyakarta,3000,4.6,120.0,"{'lat': -7.800201599999999, 'lng': 110.3663044}",-7.800202,110.366304
4,89.0,De Mata Museum Jogja,Museum De Mata merupakan salah satu museum yan...,Budaya,Yogyakarta,50000,4.4,NaN,"{'lat': -7.816315599999999, 'lng': 110.3871442}",-7.816316,110.387144


\ntour_rating: 2,870 baris × 3 kolom


,User_Id,Place_Id,Place_Ratings
0,1,101,4
1,1,154,2
2,1,103,3
3,1,208,5
4,1,89,3


\nuser: 300 baris × 3 kolom


,User_Id,Location,Age
0,1,"Semarang, Jawa Tengah",20.0
1,2,"Bekasi, Jawa Barat",21.0
2,3,"Cirebon, Jawa Barat",23.0
3,4,"Bekasi, Jawa Barat",21.0
4,5,"Lampung, Sumatera Selatan",20.0


\nPemeriksaan relasi kunci:
Place_Id unik di tour: 126
Rating dengan Place_Id yang tidak ditemukan: 0
Rating dengan User_Id yang tidak ditemukan: 0


In [4]:

# Memeriksa tipe data, data kosong, dan duplikasi

daftar_dataset = {
    "tour": df_tour,
    "tour_rating": df_rating,
    "user": df_user
}

for nama, data in daftar_dataset.items():
    print("\nDATASET:", nama)
    print("Jumlah baris dan kolom:", data.shape)

    print("\nTipe data:")
    print(data.dtypes)

    print("\nJumlah data kosong:")
    print(data.isnull().sum())

    print("\nJumlah baris duplikat:")
    print(data.duplicated().sum())



DATASET: tour
Jumlah baris dan kolom: (126, 11)

Tipe data:
Place_Id        float64
Place_Name       object
Description      object
Category         object
City             object
Price             int64
Rating          float64
Time_Minutes    float64
Coordinate       object
Latitude        float64
Longitude       float64
dtype: object

Jumlah data kosong:
Place_Id         0
Place_Name       0
Description      0
Category         0
City             0
Price            0
Rating           0
Time_Minutes    66
Coordinate       0
Latitude         0
Longitude        0
dtype: int64

Jumlah baris duplikat:
0

DATASET: tour_rating
Jumlah baris dan kolom: (2870, 3)

Tipe data:
User_Id          int64
Place_Id         int64
Place_Ratings    int64
dtype: object

Jumlah data kosong:
User_Id          0
Place_Id         0
Place_Ratings    0
dtype: int64

Jumlah baris duplikat:
23

DATASET: user
Jumlah baris dan kolom: (300, 3)

Tipe data:
User_Id       int64
Location     object
Age         float64
dty

In [6]:
# Q1.4 — Pembersihan data dan estimasi storage berdasarkan file aktual
import pandas as pd
import os

folder_processed = FOLDER_PROCESSED
os.makedirs(folder_processed, exist_ok=True)

# Salinan kerja; berkas asli pada data/raw tidak diubah.
tour_clean = df_tour.copy()
rating_clean = df_rating.copy()
user_clean = df_user.copy()

# Samakan tipe kunci agar relasi antartabel konsisten.
tour_clean["Place_Id"] = pd.to_numeric(tour_clean["Place_Id"], errors="coerce").astype("Int64")
rating_clean["Place_Id"] = pd.to_numeric(rating_clean["Place_Id"], errors="coerce").astype("Int64")
rating_clean["User_Id"] = pd.to_numeric(rating_clean["User_Id"], errors="coerce").astype("Int64")
user_clean["User_Id"] = pd.to_numeric(user_clean["User_Id"], errors="coerce").astype("Int64")

# Rapikan spasi kolom teks dan hilangkan baris duplikat identik.
for data in [tour_clean, rating_clean, user_clean]:
    for kolom in data.select_dtypes(include=["object"]).columns:
        data[kolom] = data[kolom].map(lambda x: x.strip() if isinstance(x, str) else x)

jumlah_duplikat = {
    "tour": int(tour_clean.duplicated().sum()),
    "tour_rating": int(rating_clean.duplicated().sum()),
    "user": int(user_clean.duplicated().sum())
}
tour_clean = tour_clean.drop_duplicates().copy()
rating_clean = rating_clean.drop_duplicates().copy()
user_clean = user_clean.drop_duplicates().copy()

print("HASIL PEMBERSIHAN")
for nama, data in [("tour", tour_clean), ("tour_rating", rating_clean), ("user", user_clean)]:
    print(f"{nama}: {len(data):,} baris × {len(data.columns)} kolom; duplikat tersisa={data.duplicated().sum()}")
print("Duplikat identik yang dihapus:", jumlah_duplikat)

file_tour_clean = os.path.join(folder_processed, "tour_clean.csv")
file_rating_clean = os.path.join(folder_processed, "tour_rating_clean.csv")
file_user_clean = os.path.join(folder_processed, "user_clean.csv")
tour_clean.to_csv(file_tour_clean, index=False)
rating_clean.to_csv(file_rating_clean, index=False)
user_clean.to_csv(file_user_clean, index=False)

def ukuran_file(path):
    return os.path.getsize(path) if os.path.exists(path) else 0

berkas_raw = {"tour.csv": file_tour, "tour_rating.csv": file_rating, "user.csv": file_user}
berkas_bersih = {
    "tour_clean.csv": file_tour_clean,
    "tour_rating_clean.csv": file_rating_clean,
    "user_clean.csv": file_user_clean
}
ukuran_raw = {k: ukuran_file(v) for k, v in berkas_raw.items()}
ukuran_bersih = {k: ukuran_file(v) for k, v in berkas_bersih.items()}
total_raw, total_bersih = sum(ukuran_raw.values()), sum(ukuran_bersih.values())

print("\\nESTIMASI STORAGE AKTUAL")
for nama, byte in ukuran_raw.items():
    print(f"RAW {nama}: {byte:,} byte ({byte/1024:.2f} KB)")
for nama, byte in ukuran_bersih.items():
    print(f"BERSIH {nama}: {byte:,} byte ({byte/1024:.2f} KB)")
print(f"TOTAL RAW: {total_raw:,} byte ({total_raw/1024/1024:.4f} MB)")
print(f"TOTAL BERSIH: {total_bersih:,} byte ({total_bersih/1024/1024:.4f} MB)")
print(f"RAW + BERSIH: {total_raw+total_bersih:,} byte ({(total_raw+total_bersih)/1024/1024:.4f} MB)")


HASIL PEMBERSIHAN
tour: 126 baris × 11 kolom; duplikat tersisa=0
tour_rating: 2,847 baris × 3 kolom; duplikat tersisa=0
user: 300 baris × 3 kolom; duplikat tersisa=0
Duplikat identik yang dihapus: {'tour': 0, 'tour_rating': 23, 'user': 0}
\nESTIMASI STORAGE AKTUAL
RAW tour.csv: 110,439 byte (107.85 KB)
RAW tour_rating.csv: 47,462 byte (46.35 KB)
RAW user.csv: 11,325 byte (11.06 KB)
BERSIH tour_clean.csv: 107,862 byte (105.33 KB)
BERSIH tour_rating_clean.csv: 27,146 byte (26.51 KB)
BERSIH user_clean.csv: 9,518 byte (9.29 KB)
TOTAL RAW: 169,226 byte (0.1614 MB)
TOTAL BERSIH: 144,526 byte (0.1378 MB)
RAW + BERSIH: 313,752 byte (0.2992 MB)


## Q1 — Akuisisi, pemeriksaan, pembersihan, dan estimasi storage

Notebook menggunakan arsip dari [Kaggle — Dataset Pariwisata Yogyakarta](https://www.kaggle.com/datasets/saufan/dataset-pariwisata-yogyakarta), yang berisi `tour.csv`, `tour_rating.csv`, dan `user.csv`.

Pemeriksaan terhadap arsip yang disertakan bersama notebook menemukan:
- `tour.csv`: 126 baris dan 11 kolom.
- `tour_rating.csv`: 2.870 baris dan 3 kolom; terdapat 23 baris duplikat identik.
- `user.csv`: 300 baris dan 3 kolom.
- Kolom `Time_Minutes` pada data destinasi memiliki 66 nilai kosong. Nilai kosong ini dipertahankan karena tidak boleh diisi dengan tebakan.
- Semua `Place_Id` pada rating ditemukan pada tabel destinasi, dan semua `User_Id` rating ditemukan pada tabel pengguna.

Jalankan sel upload, pembacaan, pemeriksaan, dan pembersihan secara berurutan. Data mentah tidak ditimpa; ukuran storage dihitung dari ukuran file aktual di Colab.

## Q2 — Etika data dan perlindungan privasi

Data digunakan untuk kepentingan akademik. Kolom seperti `User_Id`, `Location`, dan `Age` diperlakukan sebagai informasi yang berpotensi mengidentifikasi seseorang bila digabungkan. Data pengguna tidak dipublikasikan per individu.

Status lisensi tidak boleh dianggap sudah terverifikasi sebelum halaman dataset Kaggle benar-benar diperiksa. Checklist di bawah akan mempertahankan status **Menunggu verifikasi** untuk lisensi/ketentuan yang belum memiliki bukti.

## Q2 — Data Ethics Checklist

Sumber dataset: https://www.kaggle.com/datasets/saufan/dataset-pariwisata-yogyakarta

Berkas: `tour.csv`, `tour_rating.csv`, `user.csv`.

Lisensi: **belum diverifikasi** pada saat notebook disusun. Periksa bagian License pada halaman Kaggle dan perbarui status beserta bukti sebelum mengumpulkan tugas. Jangan mengklaim lisensi sudah diketahui jika belum ada bukti.

## Q3 — Akuisisi data cuaca melalui API Open-Meteo

Data cuaca diambil melalui endpoint arsip Open-Meteo (Archive API) karena periode pengamatan pada notebook adalah 25 Agustus–30 September 2026, yaitu periode historis relatif terhadap waktu pengerjaan. Respons JSON asli disimpan untuk menjaga provenance.

In [7]:
# Q2 — Data Ethics Checklist (berdasarkan bukti yang tersedia)
import pandas as pd
from IPython.display import display
import os

data_ethics = {
    "No.": list(range(1, 10)),
    "Pemeriksaan": [
        "Sumber data dicantumkan",
        "Lisensi dataset diperiksa",
        "Penggunaan data sesuai ketentuan lisensi",
        "Tidak mengambil data yang dilarang sumber",
        "Data pribadi/quasi-identifier diperhatikan",
        "Privasi pengguna dilindungi",
        "Data mentah dipertahankan",
        "Pembersihan dan transformasi didokumentasikan",
        "Hasil analisis disajikan secara bertanggung jawab"
    ],
    "Status": ["Ya", "Menunggu verifikasi", "Menunggu verifikasi", "Ya",
               "Ya", "Ya", "Ya", "Ya", "Ya"],
    "Bukti/Keterangan": [
        "URL Kaggle dicantumkan pada notebook",
        "Periksa bagian License pada halaman Kaggle; belum ada bukti lisensi pada notebook",
        "Konfirmasi ketentuan penggunaan setelah lisensi diverifikasi",
        "Hanya menggunakan berkas dataset yang diunduh dari sumber",
        "Memeriksa User_Id, Location, dan Age sebagai quasi-identifier",
        "Tidak mempublikasikan data pengguna per individu",
        "File sumber tetap berada di data/raw dan tidak ditimpa",
        "Kode pembersihan dan file hasil disimpan di notebook",
        "Hasil merge dan keterbatasan pencocokan dilaporkan apa adanya"
    ]
}
df_ethics = pd.DataFrame(data_ethics)
display(df_ethics)

folder_processed = FOLDER_PROCESSED if "FOLDER_PROCESSED" in globals() else "/content/proyek_uts_pariwisata/data/processed"
os.makedirs(folder_processed, exist_ok=True)
df_ethics.to_csv(os.path.join(folder_processed, "data_ethics_checklist.csv"), index=False)
print("Checklist disimpan. Status lisensi tetap menunggu sampai diverifikasi manual.")


,No.,Pemeriksaan,Status,Bukti/Keterangan
0,1,Sumber data dicantumkan,Ya,URL Kaggle dicantumkan pada notebook
1,2,Lisensi dataset diperiksa,Menunggu verifikasi,Periksa bagian License pada halaman Kaggle; be...
2,3,Penggunaan data sesuai ketentuan lisensi,Menunggu verifikasi,Konfirmasi ketentuan penggunaan setelah lisens...
3,4,Tidak mengambil data yang dilarang sumber,Ya,Hanya menggunakan berkas dataset yang diunduh ...
4,5,Data pribadi/quasi-identifier diperhatikan,Ya,"Memeriksa User_Id, Location, dan Age sebagai q..."
5,6,Privasi pengguna dilindungi,Ya,Tidak mempublikasikan data pengguna per individu
6,7,Data mentah dipertahankan,Ya,File sumber tetap berada di data/raw dan tidak...
7,8,Pembersihan dan transformasi didokumentasikan,Ya,Kode pembersihan dan file hasil disimpan di no...
8,9,Hasil analisis disajikan secara bertanggung jawab,Ya,Hasil merge dan keterbatasan pencocokan dilapo...


Checklist disimpan. Status lisensi tetap menunggu sampai diverifikasi manual.


In [8]:
# Q3.1 — Akuisisi data cuaca Open-Meteo Archive API
import requests
import pandas as pd
import time
import os

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": -7.8014,
    "longitude": 110.3647,
    "start_date": "2026-08-25",
    "end_date": "2026-09-30",
    "hourly": "temperature_2m,rain,wind_speed_10m,weather_code",
    "timezone": "Asia/Jakarta"
}
# Identitas kontak yang sudah tercantum pada notebook. Ganti jika bukan email aktif milik Anda.
CONTACT_EMAIL = "renaljono249@gmail.com"
headers = {"User-Agent": f"UTS-DAM-Informatika/1.0 (mailto:{CONTACT_EMAIL})"}

response = None
for percobaan in range(1, 6):
    try:
        response = requests.get(url, params=params, headers=headers, timeout=40)
        response.raise_for_status()
        print("STATUS HTTP:", response.status_code)
        break
    except requests.exceptions.RequestException as error:
        print(f"Percobaan {percobaan} gagal: {error}")
        if percobaan == 5:
            raise
        time.sleep(2 * percobaan)

data_api = response.json()
hourly = data_api.get("hourly", {})
kolom_wajib = ["time", "temperature_2m", "rain", "wind_speed_10m", "weather_code"]
kolom_tidak_ada = [k for k in kolom_wajib if k not in hourly]
if kolom_tidak_ada:
    raise ValueError(f"Respons API tidak memiliki kolom: {kolom_tidak_ada}")

df_api = pd.DataFrame({
    "tanggal": hourly["time"],
    "suhu": hourly["temperature_2m"],
    "curah_hujan": hourly["rain"],
    "kecepatan_angin": hourly["wind_speed_10m"],
    "kode_cuaca": hourly["weather_code"]
})

folder = FOLDER_PROCESSED if "FOLDER_PROCESSED" in globals() else "/content/proyek_uts_pariwisata/data/processed"
os.makedirs(folder, exist_ok=True)
file_csv_api = os.path.join(folder, "openmeteo_yogyakarta.csv")
df_api.to_csv(file_csv_api, index=False)

print("JUMLAH BARIS:", len(df_api))
print("JUMLAH KOLOM:", len(df_api.columns))
print("KOLOM:", list(df_api.columns))
display(df_api.head(10))


STATUS HTTP: 200
JUMLAH BARIS: 888
JUMLAH KOLOM: 5
KOLOM: ['tanggal', 'suhu', 'curah_hujan', 'kecepatan_angin', 'kode_cuaca']


,tanggal,suhu,curah_hujan,kecepatan_angin,kode_cuaca
0,2026-08-25T00:00,22.7,0.0,5.9,1
1,2026-08-25T01:00,23.2,0.0,4.7,2
2,2026-08-25T02:00,23.1,0.0,4.3,1
3,2026-08-25T03:00,23.0,0.0,4.2,1
4,2026-08-25T04:00,22.9,0.0,2.9,1
5,2026-08-25T05:00,22.7,0.0,2.0,1
6,2026-08-25T06:00,22.7,0.0,2.4,0
7,2026-08-25T07:00,24.7,0.0,2.3,0
8,2026-08-25T08:00,26.4,0.0,6.5,0
9,2026-08-25T09:00,28.2,0.0,9.9,0


In [9]:
# Q3.2 — Normalisasi JSON API menjadi tabel dan validasi
# Menggunakan DataFrame df_api yang dibuat pada sel akuisisi API.
df_normalisasi = df_api.copy()
df_normalisasi["tanggal"] = pd.to_datetime(df_normalisasi["tanggal"], errors="coerce")

print("HASIL NORMALISASI JSON")
print("Jumlah baris:", len(df_normalisasi))
print("Jumlah kolom:", len(df_normalisasi.columns))
print("Tipe data:")
print(df_normalisasi.dtypes)
display(df_normalisasi.head(10))

if len(df_normalisasi) >= 30:
    print("BERHASIL: Data memenuhi minimal 30 baris.")
else:
    print("PERHATIAN: Data kurang dari 30 baris; periksa rentang tanggal/respons API.")

file_normalisasi = os.path.join(FOLDER_PROCESSED, "hasil_normalisasi_api.csv")
df_normalisasi.to_csv(file_normalisasi, index=False)
print("File disimpan:", file_normalisasi)


HASIL NORMALISASI JSON
Jumlah baris: 888
Jumlah kolom: 5
Tipe data:
tanggal            datetime64[ns]
suhu                      float64
curah_hujan               float64
kecepatan_angin           float64
kode_cuaca                  int64
dtype: object


,tanggal,suhu,curah_hujan,kecepatan_angin,kode_cuaca
0,2026-08-25 00:00:00,22.7,0.0,5.9,1
1,2026-08-25 01:00:00,23.2,0.0,4.7,2
2,2026-08-25 02:00:00,23.1,0.0,4.3,1
3,2026-08-25 03:00:00,23.0,0.0,4.2,1
4,2026-08-25 04:00:00,22.9,0.0,2.9,1
5,2026-08-25 05:00:00,22.7,0.0,2.0,1
6,2026-08-25 06:00:00,22.7,0.0,2.4,0
7,2026-08-25 07:00:00,24.7,0.0,2.3,0
8,2026-08-25 08:00:00,26.4,0.0,6.5,0
9,2026-08-25 09:00:00,28.2,0.0,9.9,0


BERHASIL: Data memenuhi minimal 30 baris.
File disimpan: /content/proyek_uts_pariwisata/data/processed/hasil_normalisasi_api.csv


In [10]:
# Q3.3 — Provenance API: simpan JSON mentah dan checksum MD5
import json
import os
import hashlib
from datetime import datetime, timezone

folder_raw = FOLDER_RAW
folder_processed = FOLDER_PROCESSED
os.makedirs(folder_raw, exist_ok=True)
os.makedirs(folder_processed, exist_ok=True)

file_raw_api = os.path.join(folder_raw, "openmeteo_yogyakarta_raw.json")
with open(file_raw_api, "w", encoding="utf-8") as f:
    json.dump(data_api, f, indent=2, ensure_ascii=False)

def md5_file(path):
    digest = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

provenance_api = {
    "nama_dataset": "Data Cuaca Yogyakarta",
    "sumber": "Open-Meteo Archive API",
    "url_request": response.url,
    "endpoint": url,
    "metode": "GET",
    "waktu_akses_utc": datetime.now(timezone.utc).isoformat(),
    "lokasi": "Kota Yogyakarta, Indonesia",
    "latitude": params["latitude"],
    "longitude": params["longitude"],
    "periode": f'{params["start_date"]} sampai {params["end_date"]}',
    "variabel": params["hourly"].split(","),
    "status_http": int(response.status_code),
    "jumlah_baris_csv": int(len(df_api)),
    "file_json_raw": os.path.basename(file_raw_api),
    "md5_json_raw": md5_file(file_raw_api),
    "file_csv": os.path.basename(file_csv_api),
    "md5_csv": md5_file(file_csv_api),
    "nomor_eksekusi_sel_provenance": int(get_ipython().execution_count),
    "catatan_notebook": "Jalankan semua sel berurutan agar seluruh tahapan dan output tercatat."
}
file_provenance_api = os.path.join(folder_processed, "provenance_api.json")
with open(file_provenance_api, "w", encoding="utf-8") as f:
    json.dump(provenance_api, f, indent=2, ensure_ascii=False)

print("PROVENANCE API TERSIMPAN")
print(json.dumps(provenance_api, indent=2, ensure_ascii=False))


PROVENANCE API TERSIMPAN
{
  "nama_dataset": "Data Cuaca Yogyakarta",
  "sumber": "Open-Meteo Archive API",
  "url_request": "https://archive-api.open-meteo.com/v1/archive?latitude=-7.8014&longitude=110.3647&start_date=2026-08-25&end_date=2026-09-30&hourly=temperature_2m%2Crain%2Cwind_speed_10m%2Cweather_code&timezone=Asia%2FJakarta",
  "endpoint": "https://archive-api.open-meteo.com/v1/archive",
  "metode": "GET",
  "waktu_akses_utc": "2026-11-27T00:57:42.458537+00:00",
  "lokasi": "Kota Yogyakarta, Indonesia",
  "latitude": -7.8014,
  "longitude": 110.3647,
  "periode": "2026-08-25 sampai 2026-09-30",
  "variabel": [
    "temperature_2m",
    "rain",
    "wind_speed_10m",
    "weather_code"
  ],
  "status_http": 200,
  "jumlah_baris_csv": 888,
  "file_json_raw": "openmeteo_yogyakarta_raw.json",
  "md5_json_raw": "59a0e583305d8c675033df2743e051cd",
  "file_csv": "openmeteo_yogyakarta.csv",
  "md5_csv": "acf8d2e3c8907027b6c4ae216dcab60e",
  "nomor_eksekusi_sel_provenance": 10,
  "catat

In [12]:
# Q4.1 — Periksa robots.txt sebelum scraping
import requests
from urllib.robotparser import RobotFileParser

base_url = "https://visitingjogja.jogjaprov.go.id"
robots_url = base_url + "/robots.txt"
headers = {"User-Agent": "UTS-DAM-Informatika/1.0 (mailto:renaljono249@gmail.com)"}

response_robots = requests.get(robots_url, headers=headers, timeout=30)
print("STATUS HTTP:", response_robots.status_code)
print("URL ROBOTS:", robots_url)

if response_robots.status_code != 200:
    raise RuntimeError("robots.txt tidak dapat diverifikasi. Scraping dihentikan demi kepatuhan.")
print(response_robots.text[:5000])

rp = RobotFileParser()
rp.set_url(robots_url)
rp.parse(response_robots.text.splitlines())

target_url = base_url + "/wisata-jogja/"
boleh_scraping = rp.can_fetch(headers["User-Agent"], target_url)
print("\nTARGET:", target_url)
print("BOLEH SCRAPING MENURUT ROBOTS.TXT:", boleh_scraping)
if not boleh_scraping:
    print("Scraping halaman ini tidak diizinkan. Jangan lanjutkan ke sel scraping.")


STATUS HTTP: 200
URL ROBOTS: https://visitingjogja.jogjaprov.go.id/robots.txt
# START YOAST BLOCK
# ---------------------------
User-agent: *
Disallow:

Sitemap: https://visitingjogja.jogjaprov.go.id/sitemap_index.xml
# ---------------------------
# END YOAST BLOCK

TARGET: https://visitingjogja.jogjaprov.go.id/wisata-jogja/
BOLEH SCRAPING MENURUT ROBOTS.TXT: True


In [13]:
# Q4.2 — Pemeriksaan struktur halaman pertama (diagnostik)
import requests
from bs4 import BeautifulSoup

url = "https://visitingjogja.jogjaprov.go.id/wisata-jogja/"
headers = {"User-Agent": "UTS-DAM-Informatika/1.0 (mailto:renaljono249@gmail.com)"}
robots_check = requests.get("https://visitingjogja.jogjaprov.go.id/robots.txt", headers=headers, timeout=30)
robots_check.raise_for_status()
from urllib.robotparser import RobotFileParser
rp_check = RobotFileParser()
rp_check.parse(robots_check.text.splitlines())
if not rp_check.can_fetch(headers["User-Agent"], url):
    raise PermissionError("robots.txt tidak mengizinkan URL ini; permintaan dihentikan.")
response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
articles = soup.find_all("article")
titles = soup.select("h3 a, h2 a, .jeg_post_title a")
print("STATUS HTTP:", response.status_code)
print("UKURAN HTML:", len(response.content), "byte")
print("JUMLAH TAG ARTICLE:", len(articles))
print("JUMLAH LINK JUDUL:", len(titles))
for title in titles[:10]:
    print("-", title.get_text(" ", strip=True), "|", title.get("href"))


STATUS HTTP: 200
UKURAN HTML: 171480 byte
JUMLAH TAG ARTICLE: 0
JUMLAH LINK JUDUL: 17
- Waterboom Jogja Waterpark | https://visitingjogja.jogjaprov.go.id/44104/waterboom-jogja-waterpark/
- Glamping Glimpang Desa Wisata Purwosari Kulon Progo | https://visitingjogja.jogjaprov.go.id/43866/glamping-glimpang-desa-wisata-purwosari-kulon-progo/
- Menjelajah Prawirotaman, Surga Wisata Urban di Jantung Yogyakarta | https://visitingjogja.jogjaprov.go.id/43132/menjelajah-prawirotaman-surga-wisata-urban-di-jantung-yogyakarta/
- Melukat Sukma (10 Juli 2025-malam bulan purnama) | https://visitingjogja.jogjaprov.go.id/42824/melukat-sukma-10-juli-2025-malam-bulan-purnama/
- Jelajahi Space Publik di Sekitar Kotabaru: Heritage, Santai, dan Ramah Pejalan Kaki | https://visitingjogja.jogjaprov.go.id/42678/jelajahi-space-publik-di-sekitar-kotabaru-heritage-santai-dan-ramah-pejalan-kaki/
- 30 Tempat Wisata Yogyakarta Terbaru Hits 2024 | https://visitingjogja.jogjaprov.go.id/41894/30-tempat-wisata-yogyakarta

In [14]:
# Q4.3 — Pemeriksaan tautan pagination
from bs4 import BeautifulSoup
import requests

url = "https://visitingjogja.jogjaprov.go.id/wisata-jogja/"
headers = {"User-Agent": "UTS-DAM-Informatika/1.0 (mailto:renaljono249@gmail.com)"}
robots_check = requests.get("https://visitingjogja.jogjaprov.go.id/robots.txt", headers=headers, timeout=30)
robots_check.raise_for_status()
from urllib.robotparser import RobotFileParser
rp_check = RobotFileParser()
rp_check.parse(robots_check.text.splitlines())
if not rp_check.can_fetch(headers["User-Agent"], url):
    raise PermissionError("robots.txt tidak mengizinkan URL ini; permintaan dihentikan.")
response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

print("LINK PAGINATION YANG TERDETEKSI:")
for link in soup.find_all("a", href=True):
    teks, href = link.get_text(" ", strip=True), link["href"]
    if "page" in href.lower() or "next" in str(link.get("class", [])).lower() or teks.isdigit() or "berikutnya" in teks.lower():
        print("Teks:", teks, "| URL:", href)


LINK PAGINATION YANG TERDETEKSI:
Teks: 2 | URL: https://visitingjogja.jogjaprov.go.id/wisata-jogja/page/2/
Teks: 3 | URL: https://visitingjogja.jogjaprov.go.id/wisata-jogja/page/3/
Teks: 44 | URL: https://visitingjogja.jogjaprov.go.id/wisata-jogja/page/44/
Teks:  | URL: https://visitingjogja.jogjaprov.go.id/wisata-jogja/page/2/
Teks: Pagelaran Opera Tobong Jilid #2 “MINAK JINGGO NAGIH JANJI” (24 Oktober 2026) | URL: https://visitingjogja.jogjaprov.go.id/44915/pagelaran-opera-tobong-jilid-2-minak-jinggo-nagih-janji-24-oktober-2026/
Teks: DINAS | URL: https://visitingjogja.jogjaprov.go.id/?page_id=7498
Teks: PROFIL | URL: https://visitingjogja.jogjaprov.go.id/?page_id=7425
Teks: VISI & MISI | URL: https://visitingjogja.jogjaprov.go.id/?page_id=7436
Teks: STRUKTUR ORGANISASI | URL: https://visitingjogja.jogjaprov.go.id/?page_id=7441
Teks: PPID | URL: https://visitingjogja.jogjaprov.go.id/?page_id=7373


In [15]:
# Q4.4 — Uji akses halaman kedua (diagnostik)
import requests
from bs4 import BeautifulSoup

url = "https://visitingjogja.jogjaprov.go.id/wisata-jogja/page/2/"
headers = {"User-Agent": "UTS-DAM-Informatika/1.0 (mailto:renaljono249@gmail.com)"}
robots_check = requests.get("https://visitingjogja.jogjaprov.go.id/robots.txt", headers=headers, timeout=30)
robots_check.raise_for_status()
from urllib.robotparser import RobotFileParser
rp_check = RobotFileParser()
rp_check.parse(robots_check.text.splitlines())
if not rp_check.can_fetch(headers["User-Agent"], url):
    raise PermissionError("robots.txt tidak mengizinkan URL ini; permintaan dihentikan.")
response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")
print("STATUS HTTP:", response.status_code)
print("URL AKHIR:", response.url)
print("UKURAN HTML:", len(response.content), "byte")
print("JUDUL HALAMAN:", soup.title.get_text(strip=True) if soup.title else "Tidak ditemukan")


STATUS HTTP: 200
URL AKHIR: https://visitingjogja.jogjaprov.go.id/wisata-jogja/page/2/
UKURAN HTML: 173996 byte
JUDUL HALAMAN: Informasi Wisata Jogja oleh Dinas Pariwisata Daerah Istimewa Yogyakarta


In [16]:
# Q4.5 — Scraping bertahap 10 halaman + penyimpanan HTML mentah + provenance
import requests
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser
import os, time, json, hashlib
from datetime import datetime, timezone
import pandas as pd

base_url = "https://visitingjogja.jogjaprov.go.id"
robots_url = base_url + "/robots.txt"
headers = {"User-Agent": "UTS-DAM-Informatika/1.0 (mailto:renaljono249@gmail.com)"}
folder_raw = os.path.join(FOLDER_RAW, "visitingjogja_html")
folder_processed = FOLDER_PROCESSED
os.makedirs(folder_raw, exist_ok=True)
os.makedirs(folder_processed, exist_ok=True)

# Verifikasi robots.txt terlebih dahulu.
session = requests.Session()
session.headers.update(headers)
robots_response = session.get(robots_url, timeout=30)
robots_response.raise_for_status()
robot_parser = RobotFileParser()
robot_parser.set_url(robots_url)
robot_parser.parse(robots_response.text.splitlines())

def md5_bytes(data):
    return hashlib.md5(data).hexdigest()

hasil_scraping = []
catatan_halaman = []
jumlah_halaman_diminta = 10

for nomor in range(1, jumlah_halaman_diminta + 1):
    if nomor == 1:
        page_url = base_url + "/wisata-jogja/"
    else:
        page_url = f"{base_url}/wisata-jogja/page/{nomor}/"

    if not robot_parser.can_fetch(headers["User-Agent"], page_url):
        raise PermissionError(f"robots.txt melarang pengambilan: {page_url}")

    try:
        resp = session.get(page_url, timeout=35)
        resp.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Halaman {nomor} gagal diakses: {e}")
        catatan_halaman.append({
            "halaman": nomor, "url": page_url, "status_http": None,
            "jumlah_byte": 0, "md5_html": None, "status": f"Gagal: {e}"
        })
        continue

    # Simpan HTML mentah sebelum parsing.
    html_bytes = resp.content
    nama_html = f"halaman_{nomor:02d}.html"
    path_html = os.path.join(folder_raw, nama_html)
    with open(path_html, "wb") as f:
        f.write(html_bytes)

    soup = BeautifulSoup(resp.text, "html.parser")
    elemen_judul = soup.select("h3 a, h2 a, .jeg_post_title a, article a[rel='bookmark']")
    sudah = set()
    jumlah_record = 0

    for elemen in elemen_judul:
        judul = elemen.get_text(" ", strip=True)
        href = elemen.get("href")
        if not judul or not href:
            continue
        kunci = (judul, href)
        if kunci in sudah:
            continue
        sudah.add(kunci)
        hasil_scraping.append({
            "judul_wisata": judul,
            "url_wisata": href,
            "halaman_sumber": nomor,
            "url_halaman_sumber": page_url,
            "file_html_mentah": nama_html
        })
        jumlah_record += 1

    catatan_halaman.append({
        "halaman": nomor,
        "url": page_url,
        "status_http": int(resp.status_code),
        "jumlah_byte": len(html_bytes),
        "md5_html": md5_bytes(html_bytes),
        "jumlah_link_judul": jumlah_record,
        "file_html": nama_html,
        "status": "Berhasil"
    })
    print(f"Halaman {nomor}: HTTP {resp.status_code}; {len(html_bytes):,} byte; {jumlah_record} judul")
    time.sleep(2)  # jeda sopan antarpermintaan

df_scraping = pd.DataFrame(hasil_scraping)
if df_scraping.empty:
    raise RuntimeError("Tidak ada judul wisata yang berhasil diekstrak. Periksa struktur situs.")

file_scraping = os.path.join(folder_processed, "visitingjogja_yogyakarta.csv")
df_scraping.drop_duplicates(subset=["judul_wisata", "url_wisata"]).to_csv(file_scraping, index=False)

provenance_scraping = {
    "sumber": "Visiting Jogja",
    "base_url": base_url,
    "robots_url": robots_url,
    "user_agent": headers["User-Agent"],
    "metode": "GET; parsing HTML dengan BeautifulSoup",
    "waktu_akses_utc": datetime.now(timezone.utc).isoformat(),
    "jumlah_halaman_diminta": jumlah_halaman_diminta,
    "jumlah_halaman_berhasil": sum(1 for x in catatan_halaman if x["status"] == "Berhasil"),
    "jumlah_record_terekstrak": int(len(df_scraping)),
    "nomor_eksekusi_sel_scraping": int(get_ipython().execution_count),
    "folder_html_mentah": "data/raw/visitingjogja_html/",
    "file_csv": os.path.basename(file_scraping),
    "md5_csv": md5_bytes(open(file_scraping, "rb").read()),
    "detail_halaman": catatan_halaman
}
file_prov_scraping = os.path.join(folder_processed, "provenance_scraping.json")
with open(file_prov_scraping, "w", encoding="utf-8") as f:
    json.dump(provenance_scraping, f, indent=2, ensure_ascii=False)

print("\nHASIL SCRAPING")
print("Halaman berhasil:", provenance_scraping["jumlah_halaman_berhasil"], "/", jumlah_halaman_diminta)
print("Jumlah data:", len(df_scraping))
print("HTML mentah tersimpan di:", folder_raw)
print("CSV tersimpan di:", file_scraping)
display(df_scraping.head(10))


Halaman 1: HTTP 200; 171,480 byte; 17 judul
Halaman 2: HTTP 200; 173,996 byte; 17 judul
Halaman 3: HTTP 200; 174,294 byte; 17 judul
Halaman 4: HTTP 200; 174,470 byte; 17 judul
Halaman 5: HTTP 200; 174,457 byte; 17 judul
Halaman 6: HTTP 200; 173,780 byte; 17 judul
Halaman 7: HTTP 200; 174,413 byte; 17 judul
Halaman 8: HTTP 200; 174,075 byte; 17 judul
Halaman 9: HTTP 200; 174,840 byte; 17 judul
Halaman 10: HTTP 200; 174,823 byte; 17 judul

HASIL SCRAPING
Halaman berhasil: 10 / 10
Jumlah data: 170
HTML mentah tersimpan di: /content/proyek_uts_pariwisata/data/raw/visitingjogja_html
CSV tersimpan di: /content/proyek_uts_pariwisata/data/processed/visitingjogja_yogyakarta.csv


,judul_wisata,url_wisata,halaman_sumber,url_halaman_sumber,file_html_mentah
0,Waterboom Jogja Waterpark,https://visitingjogja.jogjaprov.go.id/44104/wa...,1,https://visitingjogja.jogjaprov.go.id/wisata-j...,halaman_01.html
1,Glamping Glimpang Desa Wisata Purwosari Kulon ...,https://visitingjogja.jogjaprov.go.id/43866/gl...,1,https://visitingjogja.jogjaprov.go.id/wisata-j...,halaman_01.html
2,"Menjelajah Prawirotaman, Surga Wisata Urban di...",https://visitingjogja.jogjaprov.go.id/43132/me...,1,https://visitingjogja.jogjaprov.go.id/wisata-j...,halaman_01.html
3,Melukat Sukma (10 Juli 2025-malam bulan purnama),https://visitingjogja.jogjaprov.go.id/42824/me...,1,https://visitingjogja.jogjaprov.go.id/wisata-j...,halaman_01.html
4,Jelajahi Space Publik di Sekitar Kotabaru: Her...,https://visitingjogja.jogjaprov.go.id/42678/je...,1,https://visitingjogja.jogjaprov.go.id/wisata-j...,halaman_01.html
5,30 Tempat Wisata Yogyakarta Terbaru Hits 2024,https://visitingjogja.jogjaprov.go.id/41894/30...,1,https://visitingjogja.jogjaprov.go.id/wisata-j...,halaman_01.html
6,Ruang Lukisan Keraton Yogyakarta,https://visitingjogja.jogjaprov.go.id/40998/ru...,1,https://visitingjogja.jogjaprov.go.id/wisata-j...,halaman_01.html
7,Wisata Lebaran di Yogyakarta,https://visitingjogja.jogjaprov.go.id/40721/wi...,1,https://visitingjogja.jogjaprov.go.id/wisata-j...,halaman_01.html
8,MuseumKu Gerabah,https://visitingjogja.jogjaprov.go.id/40254/mu...,1,https://visitingjogja.jogjaprov.go.id/wisata-j...,halaman_01.html
9,Sentra Kerajinan Perak Kota Gede,https://visitingjogja.jogjaprov.go.id/40176/se...,1,https://visitingjogja.jogjaprov.go.id/wisata-j...,halaman_01.html


In [17]:
# Q4.6 — Validasi hasil scraping dan provenance
import pandas as pd
import json
import os

folder = FOLDER_PROCESSED
file_csv = os.path.join(folder, "visitingjogja_yogyakarta.csv")
file_provenance = os.path.join(folder, "provenance_scraping.json")

if not os.path.exists(file_csv) or not os.path.exists(file_provenance):
    raise FileNotFoundError("Jalankan sel scraping (Q4.5) terlebih dahulu.")

df_scraping = pd.read_csv(file_csv)
with open(file_provenance, "r", encoding="utf-8") as f:
    provenance = json.load(f)

print("JUMLAH RECORD SCRAPING:", len(df_scraping))
print("JUMLAH HALAMAN SUMBER:", df_scraping["halaman_sumber"].nunique())
print("JUMLAH HALAMAN BERHASIL:", provenance["jumlah_halaman_berhasil"])
print("JUMLAH HTML MENTAH:", len([f for f in os.listdir(os.path.join(FOLDER_RAW, "visitingjogja_html")) if f.endswith(".html")]))
print("\nPROVENANCE SCRAPING:")
print(json.dumps(provenance, indent=2, ensure_ascii=False))


JUMLAH RECORD SCRAPING: 107
JUMLAH HALAMAN SUMBER: 10
JUMLAH HALAMAN BERHASIL: 10
JUMLAH HTML MENTAH: 10

PROVENANCE SCRAPING:
{
  "sumber": "Visiting Jogja",
  "base_url": "https://visitingjogja.jogjaprov.go.id",
  "robots_url": "https://visitingjogja.jogjaprov.go.id/robots.txt",
  "user_agent": "UTS-DAM-Informatika/1.0 (mailto:renaljono249@gmail.com)",
  "metode": "GET; parsing HTML dengan BeautifulSoup",
  "waktu_akses_utc": "2026-11-27T01:03:08.084519+00:00",
  "jumlah_halaman_diminta": 10,
  "jumlah_halaman_berhasil": 10,
  "jumlah_record_terekstrak": 170,
  "nomor_eksekusi_sel_scraping": 16,
  "folder_html_mentah": "data/raw/visitingjogja_html/",
  "file_csv": "visitingjogja_yogyakarta.csv",
  "md5_csv": "8815a3aca0afa064278e035599819878",
  "detail_halaman": [
    {
      "halaman": 1,
      "url": "https://visitingjogja.jogjaprov.go.id/wisata-jogja/",
      "status_http": 200,
      "jumlah_byte": 171480,
      "md5_html": "8f12dc0d380fa0f161717d5fccb48b72",
      "jumlah_link_

In [18]:
# Q5.1 — Normalisasi nama destinasi untuk pencocokan data
import pandas as pd
import re
import unicodedata
import os

folder = FOLDER_PROCESSED
df_tour = pd.read_csv(os.path.join(folder, "tour_clean.csv"))
df_scraping = pd.read_csv(os.path.join(folder, "visitingjogja_yogyakarta.csv"))

KATA_UMUM = {
    "wisata", "objek", "obyek", "destinasi", "di", "dan", "the",
    "kabupaten", "kota", "yogyakarta", "jogja"
}

def normalisasi_nama(nama):
    if pd.isna(nama):
        return ""
    nama = str(nama).lower().strip()
    nama = unicodedata.normalize("NFKD", nama)
    nama = "".join(ch for ch in nama if not unicodedata.combining(ch))
    nama = re.sub(r"[^a-z0-9 ]", " ", nama)
    token = [t for t in re.sub(r"\\s+", " ", nama).split() if t not in KATA_UMUM]
    return " ".join(token).strip()

df_tour["nama_normal"] = df_tour["Place_Name"].apply(normalisasi_nama)
df_scraping["nama_normal"] = df_scraping["judul_wisata"].apply(normalisasi_nama)

print("JUMLAH DESTINASI KAGGLE:", len(df_tour))
print("JUMLAH ARTIKEL SCRAPING:", len(df_scraping))
print("NAMA UNIK KAGGLE:", df_tour["nama_normal"].nunique())
print("NAMA UNIK SCRAPING:", df_scraping["nama_normal"].nunique())
display(df_tour[["Place_Name", "nama_normal"]].head(10))
display(df_scraping[["judul_wisata", "nama_normal"]].head(10))


JUMLAH DESTINASI KAGGLE: 126
JUMLAH ARTIKEL SCRAPING: 107
NAMA UNIK KAGGLE: 126
NAMA UNIK SCRAPING: 107


,Place_Name,nama_normal
0,Taman Pintar Yogyakarta,taman pintar
1,Keraton Yogyakarta,keraton
2,Sindu Kusuma Edupark (SKE),sindu kusuma edupark ske
3,Museum Benteng Vredeburg Yogyakarta,museum benteng vredeburg
4,De Mata Museum Jogja,de mata museum
5,Kampung Wisata Taman Sari,kampung taman sari
6,Situs Warungboto,situs warungboto
7,Nol Kilometer Jl.Malioboro,nol kilometer jl malioboro
8,Gembira Loka Zoo,gembira loka zoo
9,Sumur Gumuling,sumur gumuling


,judul_wisata,nama_normal
0,Waterboom Jogja Waterpark,waterboom waterpark
1,Glamping Glimpang Desa Wisata Purwosari Kulon ...,glamping glimpang desa purwosari kulon progo
2,"Menjelajah Prawirotaman, Surga Wisata Urban di...",menjelajah prawirotaman surga urban jantung
3,Melukat Sukma (10 Juli 2025-malam bulan purnama),melukat sukma 10 juli 2025 malam bulan purnama
4,Jelajahi Space Publik di Sekitar Kotabaru: Her...,jelajahi space publik sekitar kotabaru heritag...
5,30 Tempat Wisata Yogyakarta Terbaru Hits 2024,30 tempat terbaru hits 2024
6,Ruang Lukisan Keraton Yogyakarta,ruang lukisan keraton
7,Wisata Lebaran di Yogyakarta,lebaran
8,MuseumKu Gerabah,museumku gerabah
9,Sentra Kerajinan Perak Kota Gede,sentra kerajinan perak gede


In [19]:
# Q5.2 — Membuat kandidat pencocokan nama (exact dan fuzzy)
from difflib import SequenceMatcher
import pandas as pd

nama_scraping_unik = (
    df_scraping[["judul_wisata", "url_wisata", "nama_normal"]]
    .dropna(subset=["nama_normal"])
    .drop_duplicates(subset=["judul_wisata"])
    .to_dict("records")
)

kandidat = []
for _, baris in df_tour[["Place_Id", "Place_Name", "nama_normal"]].iterrows():
    nama_kaggle = baris["Place_Name"]
    normal_kaggle = baris["nama_normal"]
    if not normal_kaggle:
        continue
    for item in nama_scraping_unik:
        normal_scrape = item["nama_normal"]
        if not normal_scrape:
            continue
        skor = 100 * SequenceMatcher(None, normal_kaggle, normal_scrape).ratio()
        if normal_kaggle == normal_scrape or skor >= 75:
            kandidat.append({
                "Place_Id": baris["Place_Id"],
                "Nama_Kaggle": nama_kaggle,
                "Kandidat_Scraping": item["judul_wisata"],
                "URL_Scraping": item["url_wisata"],
                "Skor_Kemiripan": round(skor, 2),
                "Metode": "Persis setelah normalisasi" if normal_kaggle == normal_scrape else "Fuzzy"
            })

df_kandidat = pd.DataFrame(kandidat)
if not df_kandidat.empty:
    df_kandidat = df_kandidat.sort_values(
        ["Place_Id", "Skor_Kemiripan"], ascending=[True, False]
    )
print("JUMLAH KANDIDAT (SKOR >= 75 ATAU PERSIS):", len(df_kandidat))
display(df_kandidat.head(50) if not df_kandidat.empty else df_kandidat)
print("Kandidat fuzzy harus diperiksa manual; skor bukan bukti bahwa dua nama pasti merujuk tempat yang sama.")


JUMLAH KANDIDAT (SKOR >= 75 ATAU PERSIS): 5


,Place_Id,Nama_Kaggle,Kandidat_Scraping,URL_Scraping,Skor_Kemiripan,Metode
0,93,Gembira Loka Zoo,Gembira Loka Zoo,https://visitingjogja.jogjaprov.go.id/32937/ge...,100.00,Persis setelah normalisasi
1,154,Pantai Ngrawe (Mesra),Pantai Ngrawah,https://visitingjogja.jogjaprov.go.id/39975/pa...,78.79,Fuzzy
2,155,Pantai Ngobaran,Pantai Ngrawah,https://visitingjogja.jogjaprov.go.id/39975/pa...,75.86,Fuzzy
3,186,Pantai Glagah,Pantai Ngrawah,https://visitingjogja.jogjaprov.go.id/39975/pa...,81.48,Fuzzy
4,189,Pantai Nguluran,Pantai Ngrawah,https://visitingjogja.jogjaprov.go.id/39975/pa...,75.86,Fuzzy


Kandidat fuzzy harus diperiksa manual; skor bukan bukti bahwa dua nama pasti merujuk tempat yang sama.


In [20]:
# Q5.3 — Menggabungkan semua destinasi Kaggle dengan artikel Visiting Jogja
# Gunakan left join agar seluruh 126 destinasi Kaggle tetap ada di hasil akhir.
from difflib import SequenceMatcher
import pandas as pd

AMBANG_KEMIRIPAN = 85
scrape_records = (
    df_scraping.dropna(subset=["nama_normal"])
    .drop_duplicates(subset=["judul_wisata"])
    .copy()
)

pasangan = []
for idx_tour, baris_tour in df_tour.iterrows():
    nama_norm = baris_tour.get("nama_normal", "")
    if not nama_norm:
        continue
    skor_list = []
    for idx_scrape, baris_scrape in scrape_records.iterrows():
        nama_scrape = baris_scrape["nama_normal"]
        if not nama_scrape:
            continue
        skor = 100 * SequenceMatcher(None, nama_norm, nama_scrape).ratio()
        skor_list.append((skor, idx_scrape))
    if skor_list:
        skor_terbaik, idx_terbaik = max(skor_list, key=lambda x: x[0])
        if skor_terbaik >= AMBANG_KEMIRIPAN:
            pasangan.append({
                "idx_tour": idx_tour,
                "idx_scrape": idx_terbaik,
                "skor_kemiripan": round(skor_terbaik, 2),
                "metode_pencocokan": "Persis setelah normalisasi" if skor_terbaik == 100 else "Fuzzy"
            })

# Pastikan satu artikel tidak dipasangkan ke banyak destinasi.
pasangan = sorted(pasangan, key=lambda x: x["skor_kemiripan"], reverse=True)
terpilih, tour_terpakai, scrape_terpakai = [], set(), set()
for p in pasangan:
    if p["idx_tour"] not in tour_terpakai and p["idx_scrape"] not in scrape_terpakai:
        terpilih.append(p)
        tour_terpakai.add(p["idx_tour"])
        scrape_terpakai.add(p["idx_scrape"])

df_match = pd.DataFrame(terpilih)
if df_match.empty:
    df_match = pd.DataFrame(columns=["idx_tour", "idx_scrape", "skor_kemiripan", "metode_pencocokan"])
else:
    df_match["idx_tour"] = df_match["idx_tour"].astype(int)

# Hasil pasangan dibuat sebagai tabel lookup untuk left join.
if not df_match.empty:
    match_detail = df_match.merge(
        df_tour.reset_index().rename(columns={"index": "idx_tour"})[["idx_tour", "Place_Id"]],
        on="idx_tour", how="left"
    )
    match_detail = match_detail.merge(
        scrape_records.reset_index().rename(columns={"index": "idx_scrape"})[
            ["idx_scrape", "judul_wisata", "url_wisata", "halaman_sumber", "file_html_mentah"]
        ],
        on="idx_scrape", how="left"
    )
    match_detail = match_detail.drop(columns=["idx_tour", "idx_scrape"])
else:
    match_detail = pd.DataFrame(columns=[
        "Place_Id", "skor_kemiripan", "metode_pencocokan",
        "judul_wisata", "url_wisata", "halaman_sumber", "file_html_mentah"
    ])

df_merge = df_tour.merge(match_detail, on="Place_Id", how="left", validate="one_to_one")

# Agregasi rating per destinasi, lalu gabungkan dengan Place_Id.
rating_agregat = (
    df_rating.groupby("Place_Id", as_index=False)
    .agg(rata_rata_rating_pengguna=("Place_Ratings", "mean"),
         jumlah_rating=("Place_Ratings", "count"))
)
df_merge = df_merge.merge(rating_agregat, on="Place_Id", how="left", validate="one_to_one")

print("JUMLAH DESTINASI KAGGLE:", len(df_tour))
print("JUMLAH PASANGAN SCRAPING:", len(match_detail))
print("JUMLAH BARIS SETELAH LEFT JOIN:", len(df_merge))
print("Destinasi yang belum punya pasangan artikel:", df_merge["judul_wisata"].isna().sum())
print("Ambang fuzzy:", AMBANG_KEMIRIPAN, "%")
display(df_merge[["Place_Id", "Place_Name", "judul_wisata", "skor_kemiripan",
                  "rata_rata_rating_pengguna", "jumlah_rating"]].head(20))


JUMLAH DESTINASI KAGGLE: 126
JUMLAH PASANGAN SCRAPING: 1
JUMLAH BARIS SETELAH LEFT JOIN: 126
Destinasi yang belum punya pasangan artikel: 125
Ambang fuzzy: 85 %


,Place_Id,Place_Name,judul_wisata,skor_kemiripan,rata_rata_rating_pengguna,jumlah_rating
0,85,Taman Pintar Yogyakarta,NaN,NaN,2.720000,25
1,86,Keraton Yogyakarta,NaN,NaN,3.120000,25
2,87,Sindu Kusuma Edupark (SKE),NaN,NaN,3.157895,19
3,88,Museum Benteng Vredeburg Yogyakarta,NaN,NaN,3.125000,32
4,89,De Mata Museum Jogja,NaN,NaN,2.774194,31
5,90,Kampung Wisata Taman Sari,NaN,NaN,3.478261,23
6,91,Situs Warungboto,NaN,NaN,3.413793,29
7,92,Nol Kilometer Jl.Malioboro,NaN,NaN,3.380952,21
8,93,Gembira Loka Zoo,Gembira Loka Zoo,100.0,2.791667,24
9,94,Sumur Gumuling,NaN,NaN,3.458333,24


In [21]:
# Q5.4 — Pengayaan ringkasan cuaca dan validasi hasil penggabungan
import pandas as pd
import os

folder = FOLDER_PROCESSED
file_hasil = os.path.join(folder, "hasil_merge_pariwisata_yogyakarta.csv")

# Data cuaca mewakili satu titik koordinat Kota Yogyakarta, bukan cuaca setiap destinasi.
if "df_normalisasi" in globals() and not df_normalisasi.empty:
    cuaca = df_normalisasi.copy()
    cuaca["tanggal"] = pd.to_datetime(cuaca["tanggal"], errors="coerce")
    ringkasan_cuaca = pd.DataFrame([{
        "kota_cuaca": "Yogyakarta",
        "periode_cuaca": f'{params["start_date"]} s.d. {params["end_date"]}',
        "suhu_rata_rata_c": cuaca["suhu"].mean(),
        "curah_hujan_total_mm": cuaca["curah_hujan"].sum(),
        "kecepatan_angin_rata_rata_kmh": cuaca["kecepatan_angin"].mean()
    }])
    df_merge["kota_cuaca"] = "Yogyakarta"
    df_merge = df_merge.merge(ringkasan_cuaca, on="kota_cuaca", how="left", validate="many_to_one")
else:
    print("Data cuaca tidak tersedia; kolom ringkasan cuaca tidak ditambahkan.")

df_merge.to_csv(file_hasil, index=False)

print("HASIL PENGGABUNGAN DISIMPAN:", file_hasil)
print("Jumlah baris:", len(df_merge))
print("Jumlah kolom:", len(df_merge.columns))
print("Duplikasi seluruh baris:", df_merge.duplicated().sum())
print("Duplikasi Place_Id:", df_merge["Place_Id"].duplicated().sum())
print("Destinasi dengan artikel scraping:", df_merge["judul_wisata"].notna().sum())
print("Destinasi tanpa artikel scraping:", df_merge["judul_wisata"].isna().sum())
print("\\nNilai kosong per kolom:")
print(df_merge.isnull().sum())
display(df_merge.head(15))


HASIL PENGGABUNGAN DISIMPAN: /content/proyek_uts_pariwisata/data/processed/hasil_merge_pariwisata_yogyakarta.csv
Jumlah baris: 126
Jumlah kolom: 25
Duplikasi seluruh baris: 0
Duplikasi Place_Id: 0
Destinasi dengan artikel scraping: 1
Destinasi tanpa artikel scraping: 125
\nNilai kosong per kolom:
Place_Id                           0
Place_Name                         0
Description                        0
Category                           0
City                               0
Price                              0
Rating                             0
Time_Minutes                      66
Coordinate                         0
Latitude                           0
Longitude                          0
nama_normal                        0
skor_kemiripan                   125
metode_pencocokan                125
judul_wisata                     125
url_wisata                       125
halaman_sumber                   125
file_html_mentah                 125
rata_rata_rating_pengguna          0

,Place_Id,Place_Name,Description,Category,City,Price,Rating,Time_Minutes,Coordinate,Latitude,...,url_wisata,halaman_sumber,file_html_mentah,rata_rata_rating_pengguna,jumlah_rating,kota_cuaca,periode_cuaca,suhu_rata_rata_c,curah_hujan_total_mm,kecepatan_angin_rata_rata_kmh
0,85,Taman Pintar Yogyakarta,Taman Pintar Yogyakarta (bahasa Jawa: Hanacara...,Taman Hiburan,Yogyakarta,6000,4.5,120.0,"{'lat': -7.800671500000001, 'lng': 110.3676551}",-7.800671,...,NaN,NaN,NaN,2.720000,25,Yogyakarta,2026-08-25 s.d. 2026-09-30,26.766329,18.1,6.46464
1,86,Keraton Yogyakarta,Keraton Ngayogyakarta Hadiningrat atau Keraton...,Budaya,Yogyakarta,15000,4.6,NaN,"{'lat': -7.8052845, 'lng': 110.3642031}",-7.805284,...,NaN,NaN,NaN,3.120000,25,Yogyakarta,2026-08-25 s.d. 2026-09-30,26.766329,18.1,6.46464
2,87,Sindu Kusuma Edupark (SKE),Sindu Kusuma Edupark (SKE) merupakan sebuah de...,Taman Hiburan,Yogyakarta,20000,4.2,120.0,"{'lat': -7.767297300000001, 'lng': 110.3542486}",-7.767297,...,NaN,NaN,NaN,3.157895,19,Yogyakarta,2026-08-25 s.d. 2026-09-30,26.766329,18.1,6.46464
3,88,Museum Benteng Vredeburg Yogyakarta,Museum Benteng Vredeburg (bahasa Jawa: ???????...,Budaya,Yogyakarta,3000,4.6,120.0,"{'lat': -7.800201599999999, 'lng': 110.3663044}",-7.800202,...,NaN,NaN,NaN,3.125000,32,Yogyakarta,2026-08-25 s.d. 2026-09-30,26.766329,18.1,6.46464
4,89,De Mata Museum Jogja,Museum De Mata merupakan salah satu museum yan...,Budaya,Yogyakarta,50000,4.4,NaN,"{'lat': -7.816315599999999, 'lng': 110.3871442}",-7.816316,...,NaN,NaN,NaN,2.774194,31,Yogyakarta,2026-08-25 s.d. 2026-09-30,26.766329,18.1,6.46464
5,90,Kampung Wisata Taman Sari,Taman Sari Yogyakarta atau Taman Sari Keraton ...,Taman Hiburan,Yogyakarta,5000,4.6,NaN,"{'lat': -7.8100673, 'lng': 110.3594581}",-7.810067,...,NaN,NaN,NaN,3.478261,23,Yogyakarta,2026-08-25 s.d. 2026-09-30,26.766329,18.1,6.46464
6,91,Situs Warungboto,Situs Warungboto atau Pesanggrahan Rejawinangu...,Taman Hiburan,Yogyakarta,0,4.4,60.0,"{'lat': -7.8102685, 'lng': 110.3931513}",-7.810269,...,NaN,NaN,NaN,3.413793,29,Yogyakarta,2026-08-25 s.d. 2026-09-30,26.766329,18.1,6.46464
7,92,Nol Kilometer Jl.Malioboro,"Walaupun hanyalah sebuah persimpangan, namun p...",Taman Hiburan,Yogyakarta,0,4.7,45.0,"{'lat': -7.8013803, 'lng': 110.3647652}",-7.801380,...,NaN,NaN,NaN,3.380952,21,Yogyakarta,2026-08-25 s.d. 2026-09-30,26.766329,18.1,6.46464
8,93,Gembira Loka Zoo,Kebun Binatang Gembira Loka biasa disebut Gemb...,Cagar Alam,Yogyakarta,60000,4.5,NaN,"{'lat': -7.806234399999999, 'lng': 110.3967977}",-7.806234,...,https://visitingjogja.jogjaprov.go.id/32937/ge...,6.0,halaman_06.html,2.791667,24,Yogyakarta,2026-08-25 s.d. 2026-09-30,26.766329,18.1,6.46464
9,94,Sumur Gumuling,Sumur Gumuling adalah salah satu tempat untuk ...,Taman Hiburan,Yogyakarta,7000,4.5,NaN,"{'lat': -7.808791100000001, 'lng': 110.3591825}",-7.808791,...,NaN,NaN,NaN,3.458333,24,Yogyakarta,2026-08-25 s.d. 2026-09-30,26.766329,18.1,6.46464


In [22]:
# Q5.5 — Metadata, provenance, dan checksum hasil penggabungan
import json
import hashlib
import os
from datetime import datetime, timezone

folder = FOLDER_PROCESSED
file_merge = os.path.join(folder, "hasil_merge_pariwisata_yogyakarta.csv")

def md5_file(path):
    digest = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

metadata = {
    "nama_dataset": "Hasil Penggabungan Data Pariwisata Yogyakarta",
    "sumber_data": [
        "Kaggle - https://www.kaggle.com/datasets/saufan/dataset-pariwisata-yogyakarta",
        "Visiting Jogja (hanya halaman yang diizinkan robots.txt)",
        "Open-Meteo Archive API (ringkasan satu titik koordinat Kota Yogyakarta)"
    ],
    "waktu_pembuatan_utc": datetime.now(timezone.utc).isoformat(),
    "metode": "Left join seluruh destinasi Kaggle; agregasi rating per Place_Id; pencocokan judul scraping satu-ke-satu",
    "ambang_kemiripan": AMBANG_KEMIRIPAN,
    "jumlah_destinasi_kaggle": int(len(df_tour)),
    "jumlah_pasangan_scraping": int(df_merge["judul_wisata"].notna().sum()),
    "jumlah_destinasi_tanpa_artikel": int(df_merge["judul_wisata"].isna().sum()),
    "jumlah_baris_hasil": int(len(df_merge)),
    "jumlah_kolom_hasil": int(len(df_merge.columns)),
    "nomor_eksekusi_sel_metadata": int(get_ipython().execution_count),
    "file_hasil": os.path.basename(file_merge),
    "md5": md5_file(file_merge),
    "catatan": "Seluruh destinasi Kaggle dipertahankan. Pencocokan fuzzy bukan bukti identitas; periksa kandidat secara manual."
}

file_metadata = os.path.join(folder, "metadata_merge.json")
with open(file_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("METADATA MERGE TERSIMPAN")
print(json.dumps(metadata, indent=2, ensure_ascii=False))


METADATA MERGE TERSIMPAN
{
  "nama_dataset": "Hasil Penggabungan Data Pariwisata Yogyakarta",
  "sumber_data": [
    "Kaggle - https://www.kaggle.com/datasets/saufan/dataset-pariwisata-yogyakarta",
    "Visiting Jogja (hanya halaman yang diizinkan robots.txt)",
    "Open-Meteo Archive API (ringkasan satu titik koordinat Kota Yogyakarta)"
  ],
  "waktu_pembuatan_utc": "2026-11-27T01:04:40.963981+00:00",
  "metode": "Left join seluruh destinasi Kaggle; agregasi rating per Place_Id; pencocokan judul scraping satu-ke-satu",
  "ambang_kemiripan": 85,
  "jumlah_destinasi_kaggle": 126,
  "jumlah_pasangan_scraping": 1,
  "jumlah_destinasi_tanpa_artikel": 125,
  "jumlah_baris_hasil": 126,
  "jumlah_kolom_hasil": 25,
  "nomor_eksekusi_sel_metadata": 22,
  "file_hasil": "hasil_merge_pariwisata_yogyakarta.csv",
  "md5": "e709fadeec2b4bcb6ee3e0334f1ea12d",
  "catatan": "Seluruh destinasi Kaggle dipertahankan. Pencocokan fuzzy bukan bukti identitas; periksa kandidat secara manual."
}


In [23]:
# Q6 — Mengemas seluruh data hasil pekerjaan untuk diunduh
import shutil
from google.colab import files
import os

zip_path = "/content/data_uts_dam_revisi"
shutil.make_archive(zip_path, "zip", BASE_DIR)
print("Arsip dibuat:", zip_path + ".zip")
files.download(zip_path + ".zip")


Arsip dibuat: /content/data_uts_dam_revisi.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>